In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time
import pandas as pd
from bs4 import BeautifulSoup

# ------------------ SETUP ------------------
options = Options()
#options.add_argument("--headless")  # comment this if you want to see browser
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")

driver = webdriver.Chrome(options=options)

url = "https://piplanapane.in/en/crops/all-crops"
driver.get(url)

time.sleep(5)  # initial load

# ------------------ AUTO SCROLL ------------------
last_height = driver.execute_script("return document.body.scrollHeight")

while True:
    # Scroll down
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)

    # Get new height
    new_height = driver.execute_script("return document.body.scrollHeight")

    if new_height == last_height:
        print("✅ Reached end of page")
        break

    last_height = new_height

# ------------------ PARSE HTML ------------------
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")

cards = soup.find_all("div", class_="p-4")

data = []

for card in cards:
    name = card.find("h3")
    category = card.find("p")
    price = card.find("span")

    crop_name = name.text.strip() if name else None
    crop_category = category.text.strip() if category else None
    crop_price = price.text.strip() if price else None

    if crop_name:
        data.append({
            "Name": crop_name,
            "Category": crop_category,
            "Price": crop_price
        })

driver.quit()

# ------------------ SAVE FILES ------------------
df = pd.DataFrame(data)

# Save CSV
df.to_csv("crops_data.csv", index=False)

# Save Excel
df.to_excel("crops_data.xlsx", index=False)

print(f"✅ Total crops scraped: {len(df)}")
print("📁 Saved as crops_data.csv and crops_data.xlsx")

✅ Reached end of page
✅ Total crops scraped: 186
📁 Saved as crops_data.csv and crops_data.xlsx
